## Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("tickets.csv")
df.head()

,text,tag
0,I cannot login to my account,account_access
1,Password reset not working,account_access
2,Account locked after multiple attempts,account_access
3,Unable to verify my email,account_access
4,Login page not loading,account_access


## Zero-shot classification using LLM

In [2]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

labels = [
    "account_access",
    "billing",
    "technical_issue",
    "refund",
    "general_query"
]

text = "Payment not working"

result = classifier(text, labels, multi_label=True)

print(result)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'Payment not working', 'labels': ['billing', 'technical_issue', 'general_query', 'account_access', 'refund'], 'scores': [0.9000176787376404, 0.8984463214874268, 0.028948141261935234, 0.005730622448027134, 0.001741655170917511]}


## Top 3 tags output

In [3]:
def get_top3(text):
    result = classifier(text, labels, multi_label=True)

    scores = list(zip(result["labels"], result["scores"]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return scores[:3]


print(get_top3("Money deducted but order not placed"))

[('technical_issue', 0.808597981929779), ('billing', 0.16051019728183746), ('general_query', 0.04071923345327377)]


## Apply to full dataset

In [4]:
df["top3"] = df["text"].apply(get_top3)

df

,text,tag,top3
0,I cannot login to my account,account_access,"[(technical_issue, 0.9604818224906921), (accou..."
1,Password reset not working,account_access,"[(technical_issue, 0.9015129804611206), (accou..."
2,Account locked after multiple attempts,account_access,"[(technical_issue, 0.8847138285636902), (billi..."
3,Unable to verify my email,account_access,"[(technical_issue, 0.9597049951553345), (accou..."
4,Login page not loading,account_access,"[(technical_issue, 0.9627677798271179), (accou..."
...,...,...,...
95,General question,general_query,"[(general_query, 0.9241152405738831), (account..."
96,Need information,general_query,"[(account_access, 0.5296198725700378), (genera..."
97,Help with settings,general_query,"[(technical_issue, 0.8224080801010132), (accou..."
98,App guide,general_query,"[(account_access, 0.4430406987667084), (genera..."


## Few-shot learning (Prompt Engineering)

In [6]:
def few_shot(text):

    prompt = f"""
Ticket: I cannot login → account_access
Ticket: Payment failed → billing
Ticket: App crash → technical_issue
Ticket: Need refund → refund
Ticket: How to change email → general_query

Ticket: {text}
Category:
"""

    result = classifier(text, labels, multi_label=True)

    scores = list(zip(result["labels"], result["scores"]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return scores[:3]


print(few_shot("Refund not received"))

[('technical_issue', 0.768173098564148), ('refund', 0.6190736293792725), ('billing', 0.6190246939659119)]


## Compare Zero-shot vs Few-shot

In [7]:
df["zero_shot"] = df["text"].apply(get_top3)
df["few_shot"] = df["text"].apply(few_shot)

df

,text,tag,top3,zero_shot,few_shot
0,I cannot login to my account,account_access,"[(technical_issue, 0.9604818224906921), (accou...","[(technical_issue, 0.9604818224906921), (accou...","[(technical_issue, 0.9604818224906921), (accou..."
1,Password reset not working,account_access,"[(technical_issue, 0.9015129804611206), (accou...","[(technical_issue, 0.9015129804611206), (accou...","[(technical_issue, 0.9015129804611206), (accou..."
2,Account locked after multiple attempts,account_access,"[(technical_issue, 0.8847138285636902), (billi...","[(technical_issue, 0.8847138285636902), (billi...","[(technical_issue, 0.8847138285636902), (billi..."
3,Unable to verify my email,account_access,"[(technical_issue, 0.9597049951553345), (accou...","[(technical_issue, 0.9597049951553345), (accou...","[(technical_issue, 0.9597049951553345), (accou..."
4,Login page not loading,account_access,"[(technical_issue, 0.9627677798271179), (accou...","[(technical_issue, 0.9627677798271179), (accou...","[(technical_issue, 0.9627677798271179), (accou..."
...,...,...,...,...,...
95,General question,general_query,"[(general_query, 0.9241152405738831), (account...","[(general_query, 0.9241152405738831), (account...","[(general_query, 0.9241152405738831), (account..."
96,Need information,general_query,"[(account_access, 0.5296198725700378), (genera...","[(account_access, 0.5296198725700378), (genera...","[(account_access, 0.5296198725700378), (genera..."
97,Help with settings,general_query,"[(technical_issue, 0.8224080801010132), (accou...","[(technical_issue, 0.8224080801010132), (accou...","[(technical_issue, 0.8224080801010132), (accou..."
98,App guide,general_query,"[(account_access, 0.4430406987667084), (genera...","[(account_access, 0.4430406987667084), (genera...","[(account_access, 0.4430406987667084), (genera..."


## Accuracy comparison

In [8]:
def top1(pred):
    return pred[0][0]

df["zero_top1"] = df["zero_shot"].apply(top1)

accuracy = (df["zero_top1"] == df["tag"]).mean()

print("Accuracy:", accuracy)

Accuracy: 0.49


## Zero-shot Top-3 Accuracy:

In [9]:
def in_top3(pred, true_tag):
    top_labels = [label for label, score in pred]
    return true_tag in top_labels

df["zero_top3_correct"] = df.apply(lambda x: in_top3(x["zero_shot"], x["tag"]), axis=1)
top3_accuracy = df["zero_top3_correct"].mean()
print("Zero-shot Top-3 Accuracy:", top3_accuracy)

Zero-shot Top-3 Accuracy: 0.93


## Few-shot Top-3 Accuracy:

In [10]:
df["few_top3_correct"] = df.apply(lambda x: in_top3(few_shot(x["text"]), x["tag"]), axis=1)
fewshot_top3_acc = df["few_top3_correct"].mean()
print("Few-shot Top-3 Accuracy:", fewshot_top3_acc)

Few-shot Top-3 Accuracy: 0.93
